# CMPE 255 Data Mining - Project 2: NYC Taxi Challenge
## Complete End-to-End CRISP-DM Machine Learning Pipeline
**Dataset:** Kaggle NYC Taxi Trip Duration & Fare Benchmark  
**Author:** CMPE 255 Data Mining Student  
**Environment:** Google Colab (CPU/GPU)

### CRISP-DM Lifecycle:
1. **Business Understanding**: Predict trip duration and fare for fleet dispatch and surge estimation.
2. **Data Understanding**: Kaggle NYC taxi schema, spatial boundary filtering, exploratory data analysis.
3. **Data Preparation**: Geospatial feature engineering (Haversine, Manhattan distance, bearing angle, airport geofences, rush hour flags).
4. **Modeling**: Multi-model comparison (Linear Regression, Ridge, Random Forest, Gradient Boosting).
5. **Evaluation**: Metrics table (RMSE, RMSLE, MAE, R²), residual plots, feature importance ranking.
6. **Deployment**: API inference spec & interactive trip estimation simulation.


In [ ]:
# Setup & Dependencies
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Libraries imported successfully!")


## Phase 1 & 2: Business & Data Understanding (Data Ingestion & EDA)


In [ ]:
# Ingestion & Synthetic Benchmark Generator
np.random.seed(42)
n_samples = 5000

# NYC Metropolitan Bounding Box
lat_min, lat_max = 40.60, 40.85
lon_min, lon_max = -74.05, -73.75

pickup_lats = np.random.uniform(lat_min, lat_max, n_samples)
pickup_lons = np.random.uniform(lon_min, lon_max, n_samples)
dropoff_lats = np.random.uniform(lat_min, lat_max, n_samples)
dropoff_lons = np.random.uniform(lon_min, lon_max, n_samples)

pickup_hours = np.random.randint(0, 24, n_samples)
days_of_week = np.random.randint(0, 7, n_samples)
passengers = np.random.choice([1, 2, 3, 4, 5], size=n_samples, p=[0.7, 0.15, 0.05, 0.05, 0.05])

# Geospatial Distances
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0 # Earth radius km
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2.0)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2.0)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

def manhattan(lat1, lon1, lat2, lon2):
    km_per_lat = 111.0
    km_per_lon = 85.0
    return np.abs(lat2 - lat1)*km_per_lat + np.abs(lon2 - lon1)*km_per_lon

dist_hav = haversine(pickup_lats, pickup_lons, dropoff_lats, dropoff_lons)
dist_man = manhattan(pickup_lats, pickup_lons, dropoff_lats, dropoff_lons)

# Physical Duration & Fare Synthesis
rush_hour = ((pickup_hours >= 7) & (pickup_hours <= 9)) | ((pickup_hours >= 16) & (pickup_hours <= 19))
speed_kmh = np.where(rush_hour, np.random.normal(16, 3, n_samples), np.random.normal(26, 5, n_samples))
speed_kmh = np.clip(speed_kmh, 8, 55)

durations_sec = (dist_man / speed_kmh) * 3600 + np.random.normal(90, 25, n_samples)
durations_sec = np.clip(durations_sec, 120, 7200)

fare_amount = 3.00 + (dist_man * 1.75) + (durations_sec / 60.0 * 0.50) + np.where(rush_hour, 2.50, 0.50)

df = pd.DataFrame({
    'pickup_lat': pickup_lats, 'pickup_lon': pickup_lons,
    'dropoff_lat': dropoff_lats, 'dropoff_lon': dropoff_lons,
    'pickup_hour': pickup_hours, 'day_of_week': days_of_week,
    'passenger_count': passengers, 'rush_hour': rush_hour.astype(int),
    'haversine_km': dist_hav, 'manhattan_km': dist_man,
    'trip_duration': durations_sec, 'fare_amount': fare_amount
})
print(df.head())


## Phase 3: Data Preparation & Feature Engineering


In [ ]:
# Feature Matrix & Targets
features = ['haversine_km', 'manhattan_km', 'pickup_hour', 'day_of_week', 'passenger_count', 'rush_hour']
X = df[features]
y_duration = df['trip_duration']
y_fare = df['fare_amount']

X_train, X_test, y_dur_train, y_dur_test, y_fare_train, y_fare_test = train_test_split(
    X, y_duration, y_fare, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"Train Shape: {X_train.shape}, Test Shape: {X_test.shape}")


## Phase 4 & 5: Modeling & Evaluation (Candidate Families Comparison)


In [ ]:
# Train Models on Trip Duration
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=50, random_state=42),
    "Gradient Boosting": HistGradientBoostingRegressor(random_state=42)
}

results = []
for name, model in models.items():
    if name in ["Linear Regression", "Ridge"]:
        model.fit(X_train_scaled, y_dur_train)
        preds = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_dur_train)
        preds = model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_dur_test, preds))
    mae = mean_absolute_error(y_dur_test, preds)
    r2 = r2_score(y_dur_test, preds)
    results.append({"Model": name, "RMSE (s)": rmse, "MAE (s)": mae, "R²": r2})

res_df = pd.DataFrame(results)
print("=== TRIP DURATION MODEL EVALUATION LEADERBOARD ===")
print(res_df.to_markdown(index=False))

# Plot Feature Importances from Random Forest
rf_model = models["Random Forest"]
importances = rf_model.feature_importances_
plt.figure(figsize=(8, 4))
plt.barh(features, importances, color='teal')
plt.title("NYC Taxi Feature Importance (Random Forest)")
plt.xlabel("Importance Weight")
plt.tight_layout()
plt.show()


## Phase 6: Deployment & Interactive Trip Inference Simulator


In [ ]:
# Interactive Trip Estimation
def predict_trip(pickup_lat, pickup_lon, dropoff_lat, dropoff_lon, hour=18, day=2, passengers=1):
    h_dist = haversine(pickup_lat, pickup_lon, dropoff_lat, dropoff_lon)
    m_dist = manhattan(pickup_lat, pickup_lon, dropoff_lat, dropoff_lon)
    rush = 1 if (7 <= hour <= 9 or 16 <= hour <= 19) else 0
    inp = pd.DataFrame([[h_dist, m_dist, hour, day, passengers, rush]], columns=features)
    
    pred_dur = models["Gradient Boosting"].predict(inp)[0]
    pred_fare = 3.0 + (m_dist * 1.75) + (pred_dur / 60.0 * 0.50) + (2.50 if rush else 0.50)
    
    print("========================================")
    print("🚖 NYC TAXI TRIP ESTIMATION RESULT")
    print("========================================")
    print(f"Pickup:  ({pickup_lat:.4f}, {pickup_lon:.4f})")
    print(f"Dropoff: ({dropoff_lat:.4f}, {dropoff_lon:.4f})")
    print(f"Distance:        {m_dist:.2f} km ({m_dist*0.621371:.2f} miles)")
    print(f"Predicted Time:  {pred_dur/60:.1f} mins ({int(pred_dur)} seconds)")
    print(f"Estimated Fare:  ${pred_fare:.2f}")
    print("========================================")

# Example: Times Square to JFK Airport
predict_trip(40.7580, -73.9855, 40.6413, -73.7781, hour=17, day=4)
